# Kalman Filter Price-Target Model — Fused MvGRW Panel

**Notebook form of `pymc_kalman_filter_pt.py`**, aligned with
`probabilistic_ml_model/pymc_models/KalmanFilterModel.py`.

The cross-sectional spine is the **fused MvGRW panel model**
(`build_fused_kalman_pt_model`):

- **Model B spine** — a diagonal (NUTS-safe) Multivariate Gaussian Random Walk over the
  `(isin, time, y_series)` response tensor with a cross-sectional baseline `mu_isin`.
- **Model A refinement** — the volatility-aware `expected_return → risk_adj_return`
  latent (with a non-centred logit-normal `achieve_prob`) *is* the GRW baseline
  `mu_isin`, and the heteroscedastic scale `sigma_isin = sigma_base · (1 + cv) / √n`
  replaces the cv-free form. The risk adjustment is keyed on **expected volatility**
  (the `feat_vol_*` term-structure mean) rather than analyst conviction.

The workflow has two halves:

| Sections    | Scope                                                                                                                        |
|-------------|------------------------------------------------------------------------------------------------------------------------------|
| **§1–§10**  | Fused cross-sectional panel — one row per ISIN from `pml.mv_pymc_kalman_pt`                                                  |
| **§11–§14** | Single-security / cohort time-series — the `KalmanFilterPriceTarget` GRW filter on the embedded `*_ago` price-target history |

**Schema is the single source of truth (`pml` schema):** MV `pml.mv_pymc_kalman_pt`,
catalogue `pml.vw_pymc_feature_catalogue WHERE model_target = 'kalman_pt'`, coords
`pml.vw_pml_df_coords`.

> **Design note.** This notebook *orchestrates* the functions defined in
> `pymc_kalman_filter_pt.py`; it does not re-implement them. The module stays the
> single source of truth, and each section here calls into it and keeps the returned
> artifact at the top level so you can inspect it interactively.

## §0 — Environment & imports

`PYTENSOR_FLAGS` must be set **before** PyTensor/PyMC are first imported. On Windows
the flag parser strips backslashes from the `cxx` path (`C:\msys64\...` →
`C:msys64...`, a nonexistent compiler that hangs the *"Compiling new CVM"* step), so we
emit the g++ path with **forward slashes**, which survive the parser. If you launched
this kernel after `. .\set_env.ps1`, the existing flag is respected; otherwise we
auto-detect g++ and set a safe value here.

In [ ]:
import os
from pathlib import Path

# Set a parser-safe PYTENSOR_FLAGS *before* importing pymc/pytensor (idempotent).
if "cxx=" not in os.environ.get("PYTENSOR_FLAGS", ""):
    _gxx = next(
        (p for p in (r"C:/msys64/ucrt64/bin/g++.exe", r"C:/msys64/mingw64/bin/g++.exe")
         if Path(p).exists()),
        "",  # empty cxx -> pure-Python / numba path (nutpie still works)
    )
    os.environ["PYTENSOR_FLAGS"] = f"floatX=float64,cxx={_gxx}"

import pytensor

print("PyTensor", pytensor.__version__, "| cxx =", repr(pytensor.config.cxx),
      "| exists:", bool(pytensor.config.cxx) and Path(pytensor.config.cxx).exists())

In [ ]:
import logging

import pymc as pm
from sqlalchemy import create_engine

# Section functions live in the module (single source of truth) — import, don't re-define.
import pymc_kalman_filter_pt as kf
from pymc_kalman_filter_pt import (
    RANDOM_SEED,
    setup_plotting, resolve_db_url,
    load_kalman_df, load_feature_catalogue, resolve_feature_roles,
    run_eda, map_state_space_features, prepare_kalman_panel_inputs,
    build_panel_model, run_prior_predictive, sample_posterior, run_posterior_predictive,
    run_diagnostics, summarize_panel_screen, export_analytics,
    run_single_isin_filter, run_single_isin_stochastic_vol,
    run_mingled_cohort_filter, run_mingled_cohort_stochastic_vol,
    run_granular_forest, run_granular_further_views,
    run_summary, run_recommendations,
    present_group_effects,
)

logging.basicConfig(level=os.environ.get("LOG_LEVEL", "INFO"))
setup_plotting()
engine = create_engine(resolve_db_url())
print("Setup complete — RANDOM_SEED =", RANDOM_SEED, "| orchestrating:", kf.__file__)

## §1 — Data load & feature-role resolution

- `load_kalman_df` — cross-sectional `pml.mv_pymc_kalman_pt` snapshot (one row per ISIN,
  filtered to `observed_pt IS NOT NULL` and next earnings in the modelling horizon).
- `load_feature_catalogue` — the `kalman_pt` rows of `pml.vw_pymc_feature_catalogue`.
- `resolve_feature_roles` — groups columns by `pymc_role` (catalogue SSOT, MV-schema
  fallback): predictors, coords, responses, classification coords, fiscal-calendar and
  day-count columns.

In [ ]:
kalman_df = load_kalman_df(engine)
feature_catalogue = load_feature_catalogue(engine)
roles = resolve_feature_roles(kalman_df, feature_catalogue)
kalman_df.head()

## §2 — Exploratory data analysis

`run_eda` renders the EDA panels through a state-space lens: drift features → the
state-transition mean (`beta` slopes), noise wideners → the measurement-noise scale
`sigma_obs`. Includes missingness overview, implied-upside ridges by industry,
coord cardinality, the winsorised `feat_*` distributional summary, the observation-noise
widener marginals, the feature collinearity heatmap, and per-group implied-upside
forests (an EDA preview of the §5 group effects).

> Skip this cell for a faster run — nothing downstream depends on it.

In [ ]:
run_eda(kalman_df, roles)

## §3 — State-space feature mapping

`map_state_space_features` maps the `feat_*` columns onto Kalman roles and returns the
drift-feature list (state-transition mean / `beta` slopes).

> **Leakage guardrail:** `feat_implied_upside = (observed_pt − last_price)/last_price` is
> a deterministic function of the response, so it must never enter the drift-predictor
> matrix. The mapping asserts this.

In [ ]:
drift_features, mapping = map_state_space_features(kalman_df)
mapping

## §4 — Fused-panel data containers

`prepare_kalman_panel_inputs` filters to log-space-usable rows and builds the
`KalmanPanelInputs`: the standardised `(isin, time, y_series)` response tensor `Y`, the
fiscal-anchor time matrix `t_scaled`, the standardised drift design matrix, the
volatility / dispersion-cv / √n noise drivers, and the categorical group-effect coords.

In [ ]:
panel = prepare_kalman_panel_inputs(kalman_df, roles, drift_features)
print("Y shape (isin, time, y_series):", panel.Y.shape)
print("response_names:", panel.response_names)
print("drift_names   :", panel.drift_names)

## §5 — Build the fused MvGRW model

`build_panel_model` wraps `build_fused_kalman_pt_model` and renders the model graph.

NUTS-inplace-safety constraints baked into the builder:

- `√n_analysts` is precomputed in NumPy and passed as `pm.Data` (never `pt.sqrt` on an
  integer container).
- `sigma_base` uses `pm.Exponential` (not `HalfNormal`) so no `Abs→Sqrt` rewrite fuses
  into an inplace `Composite` op.
- The GRW uses an explicit **diagonal** innovation parameterisation instead of
  `LKJCholeskyCov` (whose onion-method Beta→Normal chain triggers an inplace rewrite
  NUTS rejects).

Pass `robust=False` for the Normal-likelihood twin (default is the Student-t panel
likelihood, which absorbs analyst outliers).

In [ ]:
robust = True
model = build_panel_model(panel, robust=robust)
pm.model_to_graphviz(model)

## §6 — Prior predictive checks

`run_prior_predictive` draws from the prior and overlays the prior implied-upside against
the empirical `observed_pt / last_price − 1`, plus the logit-normal `achieve_prob` and
heteroscedastic `sigma_isin` priors. The predictive sampling now forces the pure-Python
linker via `compile_kwargs`, so it cannot stall on C compilation.

In [ ]:
prior_idata = run_prior_predictive(model, panel)

## §7 — Posterior inference (NUTS)

`sample_posterior` tries `nutpie → numpyro → pymc` in priority order and merges the prior
groups into the posterior for one-object downstream access.

**Revised sampler settings** (see `sample_posterior`): `draws=1000, tune=1000, chains=4,
cores=4, target_accept=0.95`. 4×1000 = 4000 draws gives ample ESS; `cores=4` runs all
chains in parallel. `target_accept=0.95` is kept for the GRW funnel geometry.

> This is the long-running cell. With the `cxx` path fixed, nutpie compiles via numba
> (no g++ stall).

In [ ]:
idata = sample_posterior(model, prior_idata)
print("Group effects fitted:", present_group_effects(idata))
idata.posterior

## §8 — Posterior predictive checks

`run_posterior_predictive` samples the fused-panel posterior predictive and draws
calibration diagnostics: an ECDF overlay of replicated vs observed standardised
responses, a per-`y_series` 94% coverage table, and (best-effort) the ArviZ PIT plot.

In [ ]:
run_posterior_predictive(model, idata, panel)

## §9 — MCMC diagnostics

`run_diagnostics` reports R-hat / ESS summaries and trace / energy diagnostics over the
fused-panel posterior.

In [ ]:
run_diagnostics(idata, panel)

## §10 — Expected price targets: posterior screen & export

- `summarize_panel_screen` → a `ScreenContext` carrying the posterior `expected_upside`
  (`eu`) and `expected_pt` (`ept`) draws over `(chain, draw, isin)`, the per-ISIN
  screening `results` table (sorted by expected upside), and the structural-TS
  Monte-Carlo `mc_summary` of risk-adjusted forward returns.
- `export_analytics` de-standardises the screen and (when `write=True`) appends it to
  `analytics.kalman_filtered_price_targets`.

In [ ]:
screen = summarize_panel_screen(idata, panel)
results = screen.results
results.head(15)

In [ ]:
# Set write=True to persist to analytics.kalman_filtered_price_targets.
kalman_results = export_analytics(idata, panel, screen, write=False)
kalman_results.head()

## §11 — Single-ISIN time-series Kalman filter (+ §11b stochastic volatility)

Switches from the cross-sectional panel to the literal `KalmanFilterPriceTarget` GRW
filter on one ISIN's embedded `*_ago` price-target history (a real `(isin, asof_date,
price_target)` panel). `run_single_isin_filter` fits the funnel-free parameterization,
smooths the latent log-price state, and forecasts to future fiscal events; `…_stochastic_vol`
re-fits with the opt-in latent log-volatility random walk + robust Student-t observations.

In [ ]:
single_ctx = run_single_isin_filter(panel.frame, engine)
run_single_isin_stochastic_vol(single_ctx)

## §12 — Mingled-ISIN earnings-window cohort filter (+ §12b stochastic volatility)

`run_mingled_cohort_filter` pools the recent-earnings-window cohort into a mingled
consensus price-target series and runs the same GRW filter; `…_stochastic_vol` adds the
time-varying volatility variant.

In [ ]:
mingled_ctx = run_mingled_cohort_filter(panel.frame, engine)
run_mingled_cohort_stochastic_vol(panel.frame, mingled_ctx)

## §13 — Granular earnings-cohort posterior-predictive forest (+ §13.1 further views)

`run_granular_forest` builds the granular earnings-cohort posterior-predictive forest
from the fused-panel posterior and screen; `run_granular_further_views` adds the
supplementary prior/posterior comparison views.

In [ ]:
forest_ctx = run_granular_forest(idata, results, panel, screen, engine)
run_granular_further_views(prior_idata, panel, screen, forest_ctx)

## §14 — Comprehensive summary & actionable recommendations

`run_summary` consolidates the cross-sectional screen, the mingled-cohort filter, and the
granular forest; `run_recommendations` turns the posterior into decision-oriented
screening calls.

In [ ]:
run_summary(results, screen, forest_ctx, mingled_ctx)
run_recommendations(idata, panel, results, screen, forest_ctx)

---
### Artifacts available at the top level

`kalman_df`, `roles`, `drift_features`, `panel`, `model`, `prior_idata`, `idata`,
`screen`, `results`, `kalman_results`, `single_ctx`, `mingled_ctx`, `forest_ctx`.

To run everything in one shot instead, call `kf.main(run_eda_section=True,
write_analytics=False, robust=True)`.